In [1]:
!pip install paddlepaddle
!pip install paddleocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 27.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [5]:
from paddleocr import PaddleOCR
import os


class PaddleOCRExtractor:
    def __init__(self):
        print("Loading PaddleOCR model...")

        self.ocr = PaddleOCR(
            lang="en",
            
            # Force CPU
            device="cpu",

            # IMPORTANT: fixes your current oneDNN error
            enable_mkldnn=False,

            # CPU threads
            cpu_threads=4,

            # Disable unnecessary document processing
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )

        print("PaddleOCR loaded successfully.")

    def extract_text(self, image_path, min_confidence=0.50):

        if not os.path.exists(image_path):
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        print(f"Running OCR on: {image_path}")

        results = self.ocr.predict(
            input=image_path,
            text_rec_score_thresh=min_confidence
        )

        extracted_text = []

        for result in results:

            # PaddleOCR Result object
            data = result.json

            # Some versions wrap results inside "res"
            if "res" in data:
                data = data["res"]

            texts = data.get("rec_texts", [])
            scores = data.get("rec_scores", [])

            for i, text in enumerate(texts):

                score = (
                    float(scores[i])
                    if i < len(scores)
                    else 0.0
                )

                print(
                    f"{text} | confidence: {score:.4f}"
                )

                if score >= min_confidence:
                    extracted_text.append(text)

        return "\n".join(extracted_text)


# ==========================================================
# TEST
# ==========================================================

if __name__ == "__main__":

    image_path = "/kaggle/input/datasets/ahmedshayan1/samples123123/1131w-pyLVaYlAk1o.webp"

    extractor = PaddleOCRExtractor()

    text = extractor.extract_text(
        image_path,
        min_confidence=0.50
    )

    print("\n")
    print("=" * 70)
    print("EXTRACTED TEXT")
    print("=" * 70)
    print(text)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


Loading PaddleOCR model...


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


PaddleOCR loaded successfully.
Running OCR on: /kaggle/input/datasets/ahmedshayan1/samples123123/1131w-pyLVaYlAk1o.webp
INVOICE | confidence: 0.9999
#1024 | confidence: 0.9993
BILLED TO: | confidence: 0.9970
Really Great Company | confidence: 0.9998
PAY TO: | confidence: 0.9980
Avery Davis | confidence: 0.9756
123 Anywhere St., Any City | confidence: 0.9998
123-456-7890 | confidence: 0.9845
Bank | confidence: 1.0000
Really Great Bank | confidence: 0.9998
Account Name | confidence: 0.9998
John Smith | confidence: 1.0000
BSB | confidence: 1.0000
000-000 | confidence: 0.9882
Account Number | confidence: 0.9997
0000 0000 | confidence: 0.9708
DESCRIPTION | confidence: 1.0000
RATE | confidence: 1.0000
HOURS | confidence: 1.0000
AMOUNT | confidence: 1.0000
Content Plan | confidence: 1.0000
$50/hr | confidence: 0.9999
4 | confidence: 1.0000
$200.00 | confidence: 0.9999
Copy Writing | confidence: 1.0000
$50/hr | confidence: 0.9999
2 | confidence: 1.0000
$100.00 | confidence: 0.9995
Website Desi

In [7]:
!pip install pymupdf python-docx